# Secondary failure-type component — multilabel experiment

The binary production model has been selected and evaluated. Train a separate model for the five overlapping supplied flags using the same six sensor inputs. Preserve inconsistent labels. This is an experimental flag estimator, not a verified diagnosis; RNF has only 11 training positives and conflicts with the primary label. Fixed threshold 0.5 is set before evaluation; rare-label recall and support are reported, including zero detections.

In [1]:
from pathlib import Path
import nbformat
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/ai4i2020.csv").exists())
notebook = nbformat.read(PROJECT_ROOT / "notebooks/04_baseline_models.ipynb", as_version=4)
for cell in notebook.cells:
    if cell.cell_type == "code" and "definitions" in cell.metadata.get("tags", []):
        exec(compile(cell.source, "baseline_definitions", "exec"), globals())
from xgboost import XGBClassifier
records, manifest = load_partitions()
X_train, y_train = get_partition(records, manifest, "train")
X_validation, y_validation = get_partition(records, manifest, "validation")

In [2]:
metadata = json.loads((MODEL_DIR / "metadata.json").read_text())
assert "artifact_hashes" in metadata, "Execute the explainability notebook first"
from sklearn.multioutput import MultiOutputClassifier
failure_pipeline = Pipeline([
    ("preprocess", make_preprocessor()),
    ("classifier", MultiOutputClassifier(RandomForestClassifier(n_estimators=200, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=4))),
])
y_types_train = records.iloc[manifest["train"]][list(FAILURE_COLUMNS)]
failure_pipeline.fit(X_train, y_types_train)
results = []
for partition in ("validation", "test"):
    X_part, _ = get_partition(records, manifest, partition)
    targets = records.iloc[manifest[partition]][list(FAILURE_COLUMNS)]
    probabilities = failure_pipeline.predict_proba(X_part)
    for index, label in enumerate(FAILURE_COLUMNS):
        results.append({"partition": partition, "label": label, **evaluate_probabilities(targets[label], probabilities[index][:,1])})
joblib.dump(failure_pipeline, MODEL_DIR / "failure_types.joblib")
(METRICS_DIR / "failure_types.json").write_text(json.dumps({"formulation":"multilabel", "threshold":0.5, "experimental":True, "results":results},indent=2) + "\n")
metadata = json.loads((MODEL_DIR / "metadata.json").read_text())
metadata["failure_type_labels"] = list(FAILURE_COLUMNS)
metadata["failure_type_threshold"] = 0.5
metadata["failure_type_limitation"] = "Experimental flag estimates; rare classes and inconsistent labels; not diagnosis"
metadata["training_ranges"] = {column: {"minimum":float(X_train[column].min()),"maximum":float(X_train[column].max())} for column in NUMERIC_COLUMNS}
metadata["artifact_hashes"]["failure_types.joblib"] = hashlib.sha256((MODEL_DIR / "failure_types.joblib").read_bytes()).hexdigest()
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata,indent=2) + "\n")
display(pd.DataFrame(results).drop(columns="confusion_matrix"))
assert len(results) == 10
print("Failure-type model stored separately from binary inference.")

,partition,label,threshold,precision,recall,f1,f2,roc_auc,pr_auc,brier_score,false_negatives,false_negative_rate,support,positive_support
0,validation,TWF,0.5,0.000000,0.000000,0.000000,0.000000,0.864353,0.064033,0.008673,12,1.000000,2000,12
1,validation,HDF,0.5,0.629630,0.772727,0.693878,0.739130,0.996990,0.832951,0.005023,5,0.227273,2000,22
2,validation,PWF,0.5,0.600000,0.750000,0.666667,0.714286,0.996346,0.703657,0.005032,4,0.250000,2000,16
3,validation,OSF,0.5,0.727273,0.761905,0.744186,0.754717,0.996054,0.858510,0.004225,5,0.238095,2000,21
4,validation,RNF,0.5,0.000000,0.000000,0.000000,0.000000,0.765565,0.050500,0.002012,3,1.000000,2000,3
5,test,TWF,0.5,0.000000,0.000000,0.000000,0.000000,0.888431,0.049811,0.006484,7,1.000000,2000,7
6,test,HDF,0.5,0.612903,0.730769,0.666667,0.703704,0.996318,0.790071,0.005735,7,0.269231,2000,26
7,test,PWF,0.5,0.500000,0.733333,0.594595,0.670732,0.993720,0.578925,0.005370,4,0.266667,2000,15
8,test,OSF,0.5,0.812500,0.650000,0.722222,0.677083,0.994596,0.834691,0.003915,7,0.350000,2000,20
9,test,RNF,0.5,0.000000,0.000000,0.000000,0.000000,0.504912,0.002869,0.003094,5,1.000000,2000,5


Failure-type model stored separately from binary inference.
